# ComfyUI no Kaggle - Vid2Vid (SD1.5 + AnimateDiff v3 + ControlNet)

Configuracao da sessao (painel lateral direito do Kaggle):
* **Accelerator:** GPU T4 x2 (ou P100). O ComfyUI usa apenas uma GPU, o que e suficiente.
* **Internet:** ON (exige verificacao da conta por telefone)
* **Persistence:** Files only

Rode as celulas em ordem: 1) setup, 2) modelos, 3) executar ComfyUI.


In [ ]:
%cd /kaggle/working

# ComfyUI (usa o PyTorch que ja vem no Kaggle; nao reinstalamos o torch)
![ ! -d ComfyUI ] && git clone https://github.com/comfyanonymous/ComfyUI
%cd /kaggle/working/ComfyUI
!git pull
!pip install -q -r requirements.txt
!pip install -q av GitPython

# ComfyUI-Manager + custom nodes do workflow vid2vid
%cd /kaggle/working/ComfyUI/custom_nodes
![ ! -d ComfyUI-Manager ] && git clone https://github.com/ltdrdata/ComfyUI-Manager
![ ! -d ComfyUI-AnimateDiff-Evolved ] && git clone https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved
![ ! -d ComfyUI-Advanced-ControlNet ] && git clone https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet
![ ! -d ComfyUI-VideoHelperSuite ] && git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite
![ ! -d comfyui_controlnet_aux ] && git clone https://github.com/Fannovel16/comfyui_controlnet_aux
!pip install -q -r ComfyUI-VideoHelperSuite/requirements.txt
!pip install -q -r comfyui_controlnet_aux/requirements.txt

# cloudflared para expor a interface (o Kaggle nao tem iframe)
!wget -q -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb
%cd /kaggle/working/ComfyUI


In [ ]:
# IMPORTANTE: use sempre URLs /resolve/ (e NAO /blob/) do Hugging Face.
%cd /kaggle/working/ComfyUI

# Remove downloads corrompidos (HTML salvo como modelo)
!find models -type f \( -name "*.safetensors" -o -name "*.ckpt" -o -name "*.pth" \) -size -1M -print -delete

# Checkpoint SD1.5 - DreamShaper 8 (~2GB)
!wget -c -q --show-progress https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors -P ./models/checkpoints/
# VAE
!wget -c -q --show-progress https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/
# AnimateDiff motion module v3
!mkdir -p models/animatediff_models
!wget -c -q --show-progress https://huggingface.co/guoyww/animatediff/resolve/main/v3_sd15_mm.ckpt -P ./models/animatediff_models/
# ControlNet SD1.5 (.safetensors fp16)
!wget -c -q --show-progress https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_openpose_fp16.safetensors -P ./models/controlnet/
!wget -c -q --show-progress https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_softedge_fp16.safetensors -P ./models/controlnet/
!wget -c -q --show-progress https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_canny_fp16.safetensors -P ./models/controlnet/

# Verificacao: nada deve ter centenas de KB
!ls -lh models/checkpoints models/vae models/animatediff_models models/controlnet


### Executar ComfyUI (cloudflared)

Procure no log a linha `This is the URL to access ComfyUI: https://....trycloudflare.com`.

In [ ]:
%cd /kaggle/working/ComfyUI
import subprocess, threading, time, socket

def tunnel_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      sock.close()
      if result == 0:
        break
  print("\nComfyUI carregado, iniciando cloudflared...\n")
  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')

threading.Thread(target=tunnel_thread, daemon=True, args=(8188,)).start()

# GPU T4/P100 de 15-16GB: sem --lowvram. Use --cuda-device 0 para fixar uma GPU.
!python main.py --dont-print-server --cuda-device 0
